# 08 · ML · Baseline do modelo: valor do pedido

**Squad 2 · Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Problema P3:** estimar o `valor_total` do pedido a partir do contexto da compra.

Primeira versão funcional do modelo, ponto de partida para as melhorias. Treina um **LightGBM Regressor** sobre a
tabela de features do notebook 07, compara com o baseline **média do ticket por categoria dominante** (MAE e R²),
explica o modelo por importância de variáveis e SHAP, grava o modelo no lake e pontua um micro-lote com o
modelo relido.

| Entrada | Saída |
|---|---|
| `squad2/grupo2/ml/features_valor_pedido` | `squad2/grupo2/ml/modelo_valor_pedido` (Delta, uma linha por versão) |

In [ ]:
%pip install -q lightgbm shap

In [ ]:
%run ./00_setup_config

## Parâmetros

O alvo é treinado em `log1p` porque o ticket tem cauda longa (poucos pedidos muito caros puxariam o ajuste); a
previsão volta para reais com `expm1` antes de medir o erro. Hiperparâmetros fixos e modestos: é o baseline.

In [ ]:
from pyspark.sql import functions as F
import lightgbm as lgb, shap, numpy as np, pandas as pd, matplotlib.pyplot as plt, json, datetime as dt
from sklearn.metrics import mean_absolute_error, r2_score

CAMINHO_FEATURES, CAMINHO_MODELO = f"{BASE}/ml/features_valor_pedido", f"{BASE}/ml/modelo_valor_pedido"
CATEGORICAS = ["categoria_dominante_idx", "metodo_pagamento_idx", "periodo_idx", "uf_idx"]
FEATURES = ["hora", "dia_semana", "fim_de_semana", "n_itens", "qtd_total", "n_categorias", "qtd_por_item", "uf_informada"] + CATEGORICAS
PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.9, random_state=42, verbose=-1)
AZUL, AZUL_CLARO = "#1F4E8C", "#5B8FC9"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.titleweight": "bold", "axes.edgecolor": "#595959", "axes.grid": True, "grid.alpha": .3})

## Leitura

A tabela tem cerca de 92 mil pedidos e 20 colunas numéricas: cabe com folga na memória do driver, e o LightGBM
treina em pandas; por isso só as colunas do modelo passam para pandas, depois de filtradas no Spark.

In [ ]:
fs = ler_delta(CAMINHO_FEATURES)
if fs is None: raise FileNotFoundError("Rode o notebook 07 antes: tabela de features nao encontrada")
df = fs.select("fonte", "conjunto", "id_pedido", "categoria_dominante", "flag_item_fora_curva", "valor_total", *FEATURES).toPandas()
df[CATEGORICAS] = df[CATEGORICAS].astype("int32")
display(df.groupby(["fonte", "conjunto"]).agg(pedidos=("id_pedido", "size"), ticket_medio=("valor_total", "mean"),
        pct_fora_curva=("flag_item_fora_curva", "mean")).round(3).reset_index())

## Cenários

| Cenário | Treino | Teste | Pergunta |
|---|---|---|---|
| a | histórico (80% mais antigo) | histórico (20% mais recente) | o modelo aprende o ticket? |
| b | histórico | stream (teste) | o que foi aprendido no histórico vale no tempo real? |
| c | stream (treino) | stream (teste) | o tempo real sozinho basta? |
| d | stream sem item fora da curva | stream sem item fora da curva | quanto os preços anômalos atrapalham? |

O corte treino/teste é temporal (vem do 07), então o teste é sempre "o futuro" em relação ao treino. O baseline
de cada cenário é a média de `valor_total` por categoria dominante **no treino daquele cenário** (categoria sem
histórico recebe a média geral).

In [ ]:
def parte(fonte, conjunto, sem_fora=False):
    d = df[(df.fonte == fonte) & (df.conjunto == conjunto)]
    return d[d.flag_item_fora_curva == 0] if sem_fora else d

def treinar(tr):
    return lgb.LGBMRegressor(**PARAMS).fit(tr[FEATURES], np.log1p(tr.valor_total), categorical_feature=CATEGORICAS)

def baseline(tr, te):
    return te.categoria_dominante.map(tr.groupby("categoria_dominante").valor_total.mean()).fillna(tr.valor_total.mean())

def metricas(y, p): return dict(mae=mean_absolute_error(y, p), r2=r2_score(y, p))

CENARIOS = {"a hist→hist": (parte("historico", "treino"), parte("historico", "teste")),
            "b hist→stream": (parte("historico", "treino"), parte("stream", "teste")),
            "c stream→stream": (parte("stream", "treino"), parte("stream", "teste")),
            "d stream→stream sem fora da curva": (parte("stream", "treino", True), parte("stream", "teste", True))}
modelos, linhas = {}, []
for nome, (tr, te) in CENARIOS.items():
    m = modelos[nome] = treinar(tr)
    mm, mb = metricas(te.valor_total, np.expm1(m.predict(te[FEATURES]))), metricas(te.valor_total, baseline(tr, te))
    linhas.append(dict(cenario=nome, n_treino=len(tr), n_teste=len(te), mae_modelo=mm["mae"], mae_baseline=mb["mae"],
                       r2_modelo=mm["r2"], r2_baseline=mb["r2"], ganho_mae_pct=100 * (1 - mm["mae"] / mb["mae"])))
resultado = pd.DataFrame(linhas).round(3)
display(resultado)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
x, w = np.arange(len(resultado)), .38
for i, (met, tit) in enumerate([("mae", "MAE (R$), menor é melhor"), ("r2", "R², maior é melhor")]):
    ax[i].bar(x - w / 2, resultado[f"{met}_modelo"], w, color=AZUL, label="LightGBM")
    ax[i].bar(x + w / 2, resultado[f"{met}_baseline"], w, color=AZUL_CLARO, label="Baseline (média por categoria)")
    ax[i].set_xticks(x, [c.split(" ")[0] for c in resultado.cenario]); ax[i].set_title(tit); ax[i].legend(fontsize=8)
plt.suptitle("Modelo × baseline por cenário (a, b, c, d)", fontweight="bold"); plt.tight_layout(); plt.show()

## Explicação do modelo principal

O modelo principal desta versão é o do cenário **a** (treino no histórico): é o que tem volume para aprender e
é o que vai pontuar o tempo real. Importância por ganho (quanto cada variável reduz o erro nas divisões) e SHAP
(contribuição média absoluta de cada variável para a previsão, em escala log), numa amostra de 2.000 pedidos do teste.

In [ ]:
PRINCIPAL = "a hist→hist"
modelo = modelos[PRINCIPAL]
imp = pd.Series(modelo.booster_.feature_importance("gain"), index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp.index, imp / imp.sum() * 100, color=AZUL); ax.set_xlabel("% do ganho total"); ax.set_title("Importância das variáveis (gain)")
plt.tight_layout(); plt.show()

amostra = CENARIOS[PRINCIPAL][1][FEATURES].sample(min(2000, len(CENARIOS[PRINCIPAL][1])), random_state=42)
valores_shap = shap.TreeExplainer(modelo.booster_).shap_values(amostra)
shap.summary_plot(valores_shap, amostra, plot_type="bar", color=AZUL, show=False)
plt.gcf().set_size_inches(9, 5); plt.xlabel("média |SHAP| (escala log do ticket)"); plt.title("SHAP: contribuição média por variável"); plt.tight_layout(); plt.show()

## Gravar o modelo

O booster vai como texto (`model_to_string`) numa tabela Delta pequena, gravada pelo DataFrame writer com as
mesmas credenciais das outras escritas no lake. Cada execução acrescenta uma versão, com data, métricas e lista
de features, para que as próximas versões possam ser comparadas com esta.

In [ ]:
anterior = ler_delta(CAMINHO_MODELO)
versao = 1 if anterior is None else anterior.agg(F.max("versao")).first()[0] + 1
met_principal = resultado.set_index("cenario").loc[PRINCIPAL]
registro = spark.createDataFrame([(versao, dt.datetime.now(), PRINCIPAL, float(met_principal.mae_modelo), float(met_principal.r2_modelo),
            float(met_principal.mae_baseline), json.dumps(FEATURES), json.dumps(PARAMS), resultado.to_json(orient="records", force_ascii=False),
            modelo.booster_.model_to_string())],
            "versao int, dt_treino timestamp, cenario string, mae double, r2 double, mae_baseline double, features string, params string, metricas_cenarios string, modelo string")
registro.write.format("delta").options(**adls_options).mode("append").save(CAMINHO_MODELO)
print(f"[ok] versao {versao} gravada em {CAMINHO_MODELO.split('.net/')[-1]} | modelo com {len(registro.first().modelo):,} caracteres")

## Pontuar micro-lote com o modelo relido

Prova de uso: relê a última versão da tabela, reconstrói o booster a partir do texto e estima o ticket de 5
pedidos do teste do tempo real, como faria a cada micro-lote. A previsão relida tem que ser igual à do modelo em memória.

In [ ]:
ultima = ler_delta(CAMINHO_MODELO).orderBy(F.desc("versao")).first()
booster, cols = lgb.Booster(model_str=ultima.modelo), json.loads(ultima.features)
lote = parte("stream", "teste").sample(5, random_state=7)
pontuado = lote[["id_pedido", "categoria_dominante", "n_itens", "qtd_total", "valor_total"]].assign(
    valor_previsto=np.expm1(booster.predict(lote[cols])).round(2))
assert np.allclose(pontuado.valor_previsto, np.expm1(modelo.predict(lote[FEATURES])).round(2)), "modelo relido difere do treinado"
print(f"versao {ultima.versao} ({ultima.cenario}, treinado em {ultima.dt_treino:%d/%m/%Y %H:%M})")
display(pontuado.assign(erro=lambda d: (d.valor_previsto - d.valor_total).round(2)))

## Resumo

Números da execução de teste (histórico 90.335 pedidos, tempo real 1.834 pedidos; corte temporal 80/20):

| Cenário | MAE modelo | MAE baseline | R² modelo | R² baseline |
|---|---|---|---|---|
| a histórico → histórico | R$ 122 | R$ 312 | 0,82 | 0,25 |
| b histórico → stream | R$ 149 | R$ 480 | 0,08 | −0,82 |
| c stream → stream | R$ 137 | R$ 191 | 0,04 | 0,00 |
| d stream → stream sem item fora da curva | R$ 85 | R$ 126 | 0,44 | 0,06 |

- **O modelo funciona e supera o baseline em todos os cenários.** No histórico o erro médio cai 61% (R$ 312 → R$ 122)
  e explica 82% da variação do ticket.
- **Quantidade no carrinho manda.** `qtd_total` responde por cerca de 76% do ganho; categoria dominante e nº de itens vêm
  em seguida. Hora, dia, pagamento e UF quase não pesam: o ticket depende do que está no carrinho, não de quando nem de onde.
- **No tempo real o R² cai.** O ticket médio do stream (R$ 275) é cerca de um terço do histórico (R$ 840), então o modelo do
  histórico erra o nível (cenário b), e 1.467 pedidos de treino são pouco (cenário c). Os itens com preço fora da curva
  (5% dos pedidos) respondem por boa parte do erro: sem eles o R² sobe de 0,04 para 0,44 (cenário d).
- **Modelo gravado e reutilizável.** A versão 1 (cenário a) está em `squad2/grupo2/ml/modelo_valor_pedido`; relida da tabela,
  pontuou 5 pedidos do tempo real com a mesma previsão do modelo em memória.

**Próximas versões:** recalibrar o nível para o tempo real (treinar com histórico + stream ou ajustar pelo ticket recente),
tratar os itens fora da curva antes do treino e ajustar hiperparâmetros com validação temporal.